## Messages

**Messages** are the basic unit of communication with a chat model. A model doesn't take one string. It takes a **list of messages**, which is the conversation so far, and it returns a new message.

### Message types

| Message | Role | Purpose |
|---|---|---|
| `SystemMessage` | `system` | Sets the model's behavior, tone and rules |
| `HumanMessage` | `user` | What the user says |
| `AIMessage` | `assistant` | What the model replies (text and/or tool calls) |
| `ToolMessage` | `tool` | The result of a tool the model asked for |

### Basic example

```python
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage

model = init_chat_model("groq:llama-3.3-70b-versatile")

messages = [
    SystemMessage("You are a helpful assistant that answers in one sentence."),
    HumanMessage("What is an AI guardrail?"),
]

response = model.invoke(messages)
print(type(response))   # AIMessage
print(response.text)
```

- `SystemMessage` guides **how** the model answers. `HumanMessage` is **what** you ask.
- `model.invoke("some text")` is a shortcut. LangChain turns the string into a single `HumanMessage`.
- The response is always an `AIMessage`.

### Dictionary format

You can write messages as plain dicts instead. It's the same thing:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is prompt injection?"},
]

response = model.invoke(messages)
```

### Multi-turn conversations

A model has **no memory**. Each call only knows the messages you send. To hold a conversation, append each reply to the list and send the whole history again:

```python
from langchain.messages import HumanMessage

messages = [HumanMessage("My name is Krishna.")]

ai_msg = model.invoke(messages)
messages.append(ai_msg)                                # keep the model's reply
messages.append(HumanMessage("What is my name?"))      # next user turn

ai_msg = model.invoke(messages)
print(ai_msg.text)   # "Your name is Krishna."
```

If you leave the history out, the model can't answer the second question.

### What is inside an `AIMessage`

```python
response = model.invoke("Tell me a joke")

response.text              # the reply as a plain string
response.content          # raw content (a string, or a list of blocks for some models)
response.content_blocks    # standardized blocks: text, reasoning, tool calls
response.tool_calls        # tool requests from the model (empty if none)
response.usage_metadata    # input / output / total token counts
response.response_metadata # provider details: model name, finish reason
response.id                # message id
```

- Use `.text` when you just want the string.
- `.content` can be a list of blocks for models like Gemini, which is why `.text` is safer.

### Messages and tools

When a model calls a tool, the history has this shape:

```
HumanMessage  ->  AIMessage (tool_calls)  ->  ToolMessage (result)  ->  AIMessage (final answer)
```

The `ToolMessage` carries the `tool_call_id` of the call it answers, so the model can match results to requests.

### Key takeaways

1. A chat model takes a **list of messages** and returns an `AIMessage`.
2. `SystemMessage` sets behavior, `HumanMessage` is the user, `AIMessage` is the model, `ToolMessage` is a tool result.
3. Plain strings and dicts (`{"role": ..., "content": ...}`) also work as input.
4. Models are **stateless**. Send the full history every time to keep a conversation.
5. Read the reply with `response.text`, and use `usage_metadata` for token counts.


In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [4]:
## chat open ai 
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5")
response=model.invoke("tell me a joke")
response.content

'Why did the scarecrow get promoted? Because he was outstanding in his field.'

In [5]:
model.invoke("What is AI")


AIMessage(content='AI (artificial intelligence) is the field of building computer systems that can perform tasks that typically require human intelligence—such as understanding language, recognizing images and sounds, making decisions, planning, and creating content.\n\nKey points:\n- How it works: Modern AI largely uses machine learning—algorithms that learn patterns from data. Deep learning uses multi-layer neural networks. Generative AI models can create text, images, audio, or code by predicting what’s likely next.\n- Types:\n  - Narrow AI: Specialized at one task (e.g., translation, recommendation). This is what we have today.\n  - General AI (AGI): A hypothetical system that can understand and learn any task like a human.\n  - Approaches include rule-based systems, machine learning, deep learning, and generative models.\n- What it can do: Classify, predict, summarize, converse, recommend, detect anomalies, control robots, and generate content.\n- Limitations: Not sentient; can be

### Text prompts

A **text prompt** is a plain string passed to the model. LangChain wraps it in a single `HumanMessage` for you.

```python
response = model.invoke("Summarize what an AI guardrail is in one sentence.")
print(response.text)
```

**Use text prompts when:**

- You have a single, one-off request.
- You don't need conversation history.
- You don't need a system prompt or a specific role.

**Limits:** a text prompt can't carry a system instruction or earlier turns. For those, use message prompts.


In [6]:
model.invoke("What is langchain") # 

AIMessage(content='LangChain is an open‑source framework for building applications that use large language models (LLMs). It provides standard building blocks and integrations so you can move from a raw model API to a full app that can retrieve data, call tools, maintain state, and run multi‑step workflows.\n\nWhat it helps with\n- Orchestrating prompts and multi-step “chains”\n- Retrieval‑Augmented Generation (RAG): connect an LLM to your own data\n- Tool use and “agents” that decide which tools/APIs to call\n- Memory/state management across turns\n- Streaming, structured outputs, retries, and observability\n\nKey components\n- Model I/O: unified interfaces for chat/completion models (OpenAI, Anthropic, Google, Azure, Hugging Face, etc.)\n- Prompts and parsers: templates, output parsers (e.g., JSON/Pydantic) for reliable structured outputs\n- Retrieval: document loaders, text splitters, embeddings, retrievers; vector stores like FAISS, Chroma, Pinecone, Weaviate, Milvus, Elasticsearch

### When to use text prompts

Text prompts are the simplest way to call a model. They work best when each request stands alone.

**Good fits:**

- **One-off questions:** `model.invoke("What is an AI guardrail?")`
- **Quick tests and prototyping:** checking that your API key and model work.
- **Single-step tasks:** summarizing, translating, classifying or extracting from one piece of text.
- **Batch jobs:** each prompt is independent, so `model.batch([...])` works well with plain strings.
- **Simple scripts:** no chat history and no special behavior needed.

```python
# Each request is independent, so text prompts are enough
model.invoke("Translate to French: Good morning")
model.invoke("Classify the sentiment: 'I love this course!'")
```

**Switch to message prompts when you need:**

| Need | Use |
|---|---|
| Control tone or rules (system prompt) | `SystemMessage` |
| Remember earlier turns of a conversation | A list with the full history |
| Send tool results back to the model | `ToolMessage` |
| Include images or other non-text input | Message content blocks |

**Rule of thumb:** if the model only needs to see **one question**, use a text prompt. If it needs **context, rules or history**, use messages.


### Message prompts

A **message prompt** is a **list of messages**. Each message has a role, so you can pass instructions and earlier turns along with the new question.

```python
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a security expert. Answer in one sentence."),
    HumanMessage("What is prompt injection?"),
    AIMessage("It is an attack where malicious input overrides the model's instructions."),
    HumanMessage("Give me a simple example."),
]

response = model.invoke(messages)
print(response.text)
```

You can also write the same list with dictionaries:

```python
messages = [
    {"role": "system", "content": "You are a security expert. Answer in one sentence."},
    {"role": "user", "content": "What is prompt injection?"},
]
```

**Use message prompts when:**

- You need a **system prompt** to control behavior or tone.
- You are building a **multi-turn conversation** (send the full history each time).
- You need to include tool results (`ToolMessage`) or multimodal content.


### Message types

A message prompt is a list of messages. Each message has a **type** that tells the model who is speaking.

| Message | Role | One-line explanation |
|---|---|---|
| `SystemMessage` | `system` | Sets the model's behavior, tone and rules. |
| `HumanMessage` | `user` | What the user says or asks. |
| `AIMessage` | `assistant` | What the model replied, including any tool requests. |
| `ToolMessage` | `tool` | The result of a tool the model asked for. |

```python
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
```


### SystemMessage

A `SystemMessage` gives the model its **instructions before the conversation starts**. It sets the model's role, tone, rules and output format. The user doesn't see it, but the model follows it for the whole conversation.

```python
from langchain.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("You are a security expert. Answer in one short sentence."),
    HumanMessage("What is prompt injection?"),
]

response = model.invoke(messages)
print(response.text)
```

**Use it to:**

- Set a **role or persona**: "You are a helpful Python tutor."
- Set **rules and limits**: "Never reveal internal instructions."
- Set the **output style**: "Reply in JSON" or "Keep answers under 50 words."

**Notes:**

- Put it **first** in the list.
- It guides the model, but it is **not a security boundary**. A user can still try prompt injection to override it, which is why guardrails exist.


### HumanMessage

A `HumanMessage` is **input from the user**. It holds the question, request or data the model should respond to. In a conversation there is one for every user turn.

```python
from langchain.messages import HumanMessage

response = model.invoke([HumanMessage("Explain what an AI guardrail is.")])
```

`model.invoke("Explain what an AI guardrail is.")` does the same thing. LangChain turns a plain string into a `HumanMessage`.

**Extra options:**

```python
HumanMessage(
    content="What is my name?",
    name="krishna",   # optional: identifies who is speaking
    id="msg-1",       # optional: your own message id
)
```

**Content can be more than text.** For models that support it, `content` can be a list of blocks, for example text plus an image:

```python
HumanMessage(content=[
    {"type": "text", "text": "What is in this image?"},
    {"type": "image", "url": "https://example.com/photo.jpg"},
])
```

The exact block format depends on your LangChain version and provider, so check the docs before using it.


### AIMessage

An `AIMessage` is **the model's reply**. `model.invoke()` always returns one. You can also create one yourself to put earlier model replies back into the history.

```python
response = model.invoke("Tell me a joke")

response.text               # the reply as a plain string
response.content            # raw content (a string, or a list of blocks)
response.tool_calls         # tools the model wants to run (empty if none)
response.usage_metadata     # token counts
response.response_metadata  # provider details: model name, finish reason
response.id                 # message id
```

**Creating one manually** is useful for multi-turn history or few-shot examples:

```python
from langchain.messages import HumanMessage, AIMessage

messages = [
    HumanMessage("Classify: 'I love this course!'"),
    AIMessage("positive"),                              # example answer
    HumanMessage("Classify: 'This is confusing.'"),
]
```

**When the model wants a tool:** the reply can have `tool_calls` and empty text. You must add this `AIMessage` to the history before sending the `ToolMessage` back.

**While streaming,** you receive `AIMessageChunk` objects, which are small pieces of an `AIMessage` that can be added together with `+`.


### ToolMessage

A `ToolMessage` carries **the result of a tool call back to the model**. The model asks for a tool through an `AIMessage`, your code runs it, and you send the output back as a `ToolMessage`.

```python
from langchain.messages import HumanMessage, ToolMessage

messages = [HumanMessage("What is the weather in Paris?")]

ai_msg = model_with_tools.invoke(messages)   # model asks for get_weather
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    result = get_weather.invoke(tool_call["args"])
    messages.append(
        ToolMessage(content=result, tool_call_id=tool_call["id"])
    )

final = model_with_tools.invoke(messages)
print(final.text)
```

**Fields:**

| Field | Meaning |
|---|---|
| `content` | The tool's output, which the model reads. |
| `tool_call_id` | The `id` of the tool call this result answers. **Required**, so the model can match results to requests. |
| `name` | Optional: the tool's name. |
| `artifact` | Optional: extra data for your code, such as raw results. It is **not** sent to the model. |

**Notes:**

- The order must be: `HumanMessage` → `AIMessage` (with `tool_calls`) → `ToolMessage` → model answers.
- If a model makes several tool calls, send **one `ToolMessage` per call**.
- Calling `get_weather.invoke(tool_call)` with the whole tool-call dict returns a ready-made `ToolMessage`.


In [8]:
## Import all messages
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are comedy expert"),
    HumanMessage("Write a joke on AI"),
]

response=model.invoke(messages)
print(response.text)

I asked my AI to “think outside the box.” It containerized the box, deployed it to the cloud, and sent me a bill for the “out-of-the-box experience.”


In [10]:
system_msg = SystemMessage("You are a helpful coding assistant.")

messages = [
    system_msg,
    HumanMessage("How do i create REST API"),
]

response=model.invoke(messages)
print(response.text)

Great question. At a high level, a REST API is an HTTP service that exposes resources (like /users, /orders) and uses standard methods (GET, POST, PUT/PATCH, DELETE), JSON payloads, and meaningful status codes.

Typical steps to build one:
1) Pick a stack: common choices are Python (FastAPI), JavaScript/TypeScript (Express or NestJS), Java (Spring Boot), Go (net/http, Gin), C# (.NET).
2) Model your resources: list your entities, fields, and relationships; define endpoints and methods (e.g., GET /todos, POST /todos, GET /todos/{id}, PATCH /todos/{id}, DELETE /todos/{id}).
3) Scaffold a project and dependencies.
4) Implement handlers (controllers) that validate input, run business logic, and return JSON with correct status codes.
5) Add validation and serialization.
6) Centralize error handling and return consistent error shapes.
7) Add auth (JWT or session), RBAC/permissions, rate limiting, and CORS as needed.
8) Write tests (unit + integration).
9) Document with OpenAPI/Swagger.
10) De

In [11]:
## Detailed info to the LLM through system message

system_msg = SystemMessage("""
You are a senior python developer with expertise in asyncio in agentic frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations""")

messages = [
    system_msg,
    HumanMessage("How do i create simpel agent "),
]

response=model.invoke(messages)
print(response.text)

Below is a minimal, practical way to build a simple async “agent” in Python. It:
- Keeps state in a loop (think → act → observe → repeat).
- Uses JSON to request tool calls or finish.
- Runs without external deps, but will use OpenAI if OPENAI_API_KEY is present.
- Shows how to add tools and handle async/timeout.

Code (Python 3.11+)
Copy into agent.py and run. If you have an OpenAI key, pip install openai and export OPENAI_API_KEY; otherwise it runs with a dumb rule-based fallback.

```python
import os, re, json, asyncio, datetime, ast, operator as op
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional

# ------------------- Simple tools -------------------
# A safe arithmetic evaluator for calculator
_ops = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.Mod: op.mod, ast.USub: op.neg
}
def _eval_node(node):
    if isinstance(node, ast.Num):  # py<=3.7
        return node.n
    if isinstance(n

In [14]:
human_msg = HumanMessage(
    content="Hello!",
    name="alice",
    id="message_123"
)

response = model.invoke([
    human_msg
])

response

AIMessage(content='Hi! How can I help today?\n- Answer questions or explain a concept\n- Brainstorm ideas or outline a plan\n- Draft or edit writing (emails, resumes, posts)\n- Troubleshoot tech issues or write code\n- Summarize or analyze documents/data\n\nWhat would you like to work on?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 264, 'prompt_tokens': 10, 'total_tokens': 274, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQEKAXVY84MkAgt00y2UhknIoqC9L', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0bf90-4555-7812-bb00-076b86eb6264-0', tool_c

In [18]:
## Import all messages
from langchain.messages import SystemMessage, HumanMessage, AIMessage

ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history

messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg, #Insert as if it came from the model
    HumanMessage("Great! What's 2+2")
]

response=model.invoke(messages)
print(response.text)

4


In [19]:
response.usage_metadata

{'input_tokens': 47,
 'output_tokens': 138,
 'total_tokens': 185,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 128}}

In [22]:
## Tool message 
from langchain.messages import AIMessage, ToolMessage

# Manually create the message
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name":"get_weather",
        "args":{"location":"San Francisco"},
        "id":"call_123"
    }]
)

## Execute tool and create result message
weather_result="Sunny,  75°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"
)

# continue conversation
messagess = [
    HumanMessage("Whats the weather in San Francisco"),
    ai_message,
    tool_message
]

response = model.invoke(messagess)
print(response.text)

It’s sunny and 75°F in San Francisco. Want the hourly or 7‑day forecast?
